In [1]:
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from utilities3 import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from setups import Dataset

import math


matplotlib.rcParams.update({
    'font.size': 16,           # dimensione base
    'axes.titlesize': 20,      # titoli dei subplot
    'axes.labelsize': 16,      # etichette assi
    'xtick.labelsize': 16,     # tick x
    'ytick.labelsize': 16,     # tick y
    'legend.fontsize': 16,     # legende
    'figure.titlesize': 20,    # titolo generale
})

modes = 24
width = 64
T = 10
epochs  =100

#######################################################################
############################ One Stage Spline #########################
#######################################################################

path_onespline = f'Spline_One_Stage_Adam_ep{epochs}_m{modes}_w{width}_T{T}'

path_model_onespline     = os.path.join('model',  path_onespline)
path_image_onespline     = os.path.join('image',  path_onespline)
path_loss_dir_onespline  = os.path.join('loss',   path_onespline)

#######################################################################
############################ Multi Stage Spline #######################
#######################################################################

path_multispline = f'Spline_Multi_Stage_Adam_ep{epochs}_m{modes}_w{width}_T{T}'

path_model_multispline     = os.path.join('model',  path_multispline)
path_image_multispline     = os.path.join('image',  path_multispline)
path_loss_dir_multispline  = os.path.join('loss',   path_multispline)

#######################################################################
############################ Spline superv #######################
#######################################################################

path_spline_sup = f'Spline_sup_Adam_ep{epochs}_m{modes}_w{width}_T{T}'

path_model_splinesup     = os.path.join('model',  path_spline_sup)
path_image_splinesup     = os.path.join('image',  path_spline_sup)
path_loss_dir_splinesup = os.path.join('loss',   path_spline_sup)


#######################################################################
############################ One Stage FNO ############################
#######################################################################

path_onefno = f'FNO_One_Stage_ep{epochs}_m{modes}_w{width}_T{T}'

path_model_onefno      = os.path.join('model',  path_onefno )
path_image_onefno      = os.path.join('image',  path_onefno )
path_loss_dir_onefno   = os.path.join('loss',   path_onefno )

#######################################################################
############################ Multi Stage FNO ############################
#######################################################################

path_multifno = f'FNO_Multi_Stage_ep{epochs}_m{modes}_w{width}_T{T}'

path_model_multifno     = os.path.join('model',  path_multifno)
path_image_multifno     = os.path.join('image',  path_multifno)
path_loss_dir_multifno  = os.path.join('loss',   path_multifno)

#######################################################################
############################ Supervised FNO ############################
#######################################################################
epochs = 250
path_supervised = f'FNO_Sup_ep{epochs}_m{modes}_w{width}_T{T}'
path_model_supervised     = os.path.join('model',  path_supervised)
path_image_supervised     = os.path.join('image',  path_supervised)
path_loss_dir_supervised  = os.path.join('loss',   path_supervised)

PHASE_SPLIT_X = 200

In [2]:
def load_test_full_series(loss_dir, phases_list, last_mean=True, win=11, log_space=True):
    """
    Carica SOLO 'loss_full' dai CSV per-fase.
    Ritorna (x_concat, y_concat_raw, last_val) dove last_val è:
      - l'ultimo valore della media (rolling) se last_mean=True
      - altrimenti l'ultimo grezzo
    """
    cum_epoch = 0
    xs, ys = [], []
    last_val = None
    found_any = False

    for ph in phases_list:
        f = os.path.join(loss_dir, f'loss_{ph}.csv')
        if not os.path.isfile(f):
            continue
        df = pd.read_csv(f, index_col='epoch_in_phase')
        if 'loss_full' not in df.columns:
            continue

        y = df['loss_full'].to_numpy()
        x = np.arange(1, len(y)+1) + cum_epoch

        xs.append(x)
        ys.append(y)
        cum_epoch += len(y)
        found_any = True

        # ultimo valore (media o raw) della **fase corrente**;
        # alla fine rimarrà quello dell'ultima fase trovata
        if last_mean:
            if log_space:
                ymean, _, _ = rolling_stats_log(y, win=win)
            else:
                ymean, _ = rolling_stats(y, win=win)
            last_val = float(ymean[-1]) if len(y) else None
        else:
            last_val = float(y[-1]) if len(y) else None

    if found_any:
        return np.concatenate(xs), np.concatenate(ys), last_val

    # debug opzionale
    try:
        print(f"⚠️  In {loss_dir} I see files:", os.listdir(loss_dir))
    except Exception:
        pass
    return None, None, None

def final_plateau_stats(y: np.ndarray, rel_tol: float = 2e-3,   # 0.2% variazione relativa
                        min_len: int = 20, tail_max: int = 100):
    """
    Restituisce (mean, std, i0, i1) della porzione finale (plateau) di y.
    Criterio: partendo dalla fine, accumula punti finché |Δy|/|y| <= rel_tol.
    Se la finestra trovata è corta (< min_len), usa gli ultimi 'tail_max' punti.
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n == 0:
        return np.nan, np.nan, 0, 0

    eps = 1e-12
    dy_rel = np.abs(np.diff(y)) / (np.abs(y[:-1]) + eps)

    cnt = 1  # include sempre l'ultimo punto
    for k in range(n-2, -1, -1):
        if dy_rel[k] <= rel_tol:
            cnt += 1
        else:
            break

    L = max(cnt, min_len)
    if L < min_len:
        L = min(tail_max, n)

    i0 = n - L
    i1 = n
    seg = y[i0:i1]
    return float(np.mean(seg)), float(np.std(seg)), i0, i1


def load_boundary(loss_dir, phases_list, last_mean=True, win=11, log_space=True):
    """
    Come sopra ma per 'loss_boundary'.
    """
    cum_epoch = 0
    xs, ys = [], []
    last_val = None
    found_any = False

    for ph in phases_list:
        f = os.path.join(loss_dir, f'loss_{ph}.csv')
        if not os.path.isfile(f):
            continue
        df = pd.read_csv(f, index_col='epoch_in_phase')
        if 'loss_boundary' not in df.columns:
            continue

        y = df['loss_boundary'].to_numpy()
        x = np.arange(1, len(y)+1) + cum_epoch

        xs.append(x)
        ys.append(y)
        cum_epoch += len(y)
        found_any = True

        if last_mean:
            if log_space:
                ymean, _, _ = rolling_stats_log(y, win=win)
            else:
                ymean, _ = rolling_stats(y, win=win)
            last_val = float(ymean[-1]) if len(y) else None
        else:
            last_val = float(y[-1]) if len(y) else None

    if found_any:
        return np.concatenate(xs), np.concatenate(ys), last_val

    try:
        print(f"⚠️  In {loss_dir} I see files:", os.listdir(loss_dir))
    except Exception:
        pass
    return None, None, None

import numpy as np

def rolling_stats(y, win=11):
    y = np.asarray(y, dtype=float)
    if len(y) == 0: return y, y
    if win % 2 == 0: win += 1
    pad = win // 2
    ypad = np.pad(y, (pad, pad), mode='edge')
    k = np.ones(win, dtype=float) / win
    mean = np.convolve(ypad, k, mode='valid')
    sq   = np.convolve(ypad**2, k, mode='valid')
    std  = np.sqrt(np.maximum(0.0, sq - mean**2))
    return mean, std

def rolling_stats_log(y, win=11, eps=1e-12):
    """Media/std su log10(y); ritorna (mean, lower, upper) in spazio lineare."""
    y = np.asarray(y, dtype=float)
    y = np.maximum(y, eps)
    m_log, s_log = rolling_stats(np.log10(y), win=win)
    lower = 10**(m_log - s_log)
    upper = 10**(m_log + s_log)
    mean  = 10**m_log
    return mean, lower, upper

def plot_with_band_per_phases(ax, x, y, color, label, phase_lengths, win=11):
    """Disegna per-segmento (una curva continua non attraversa i ‘gradoni’)."""
    start = 0
    first = True
    for n in phase_lengths:
        xs = x[start:start+n]; ys = y[start:start+n]
        if len(xs) == 0: 
            start += n; 
            continue
        ymean, ylow, yup = rolling_stats_log(ys, win=win)   # banda in log-space
        # clamp per asse log
        eps = 1e-12
        ylow = np.maximum(ylow, eps)

        ax.plot(xs, ymean, color=color, linewidth=2.0, label=label if first else None, zorder=3)
        ax.fill_between(xs, ylow, yup, color=color, alpha=0.18, linewidth=0, zorder=2)
        first = False
        start += n


In [3]:
#######################################################################
############################  MultiStage vs Single stage Spline #######
#######################################################################
import os
import numpy as np
import matplotlib.pyplot as plt


phases = ['P1_border_only', 'P2_res_warm']
fields = ['loss_boundary', 'loss_full', 'loss_res']

x_spline_multi, y_spline_multi, last_spline_multi = load_test_full_series(path_loss_dir_multispline, phases)
x_spline_one,   y_spline_one,   last_spline_one   = load_test_full_series(path_loss_dir_onespline,   phases)
x_spline_sup,   y_spline_sup,   last_spline_sup   = load_test_full_series(path_loss_dir_splinesup,   phases)
BLUE  =  "#7C4DFF"
GREEN = "#00E676"

# plot
plt.figure(figsize=(8,5))
plt.yscale('log')
ax = plt.gca()
plotted_any = False

# --- Multi-Stage con banda (log-space) ---
if x_spline_multi is not None and y_spline_multi is not None:
    ymean_m, lower_m, upper_m = rolling_stats_log(y_spline_multi, win=11)
    ax.plot(x_spline_multi, ymean_m, label='M-S', color=GREEN, linewidth=2.0, zorder=3)
    ax.fill_between(x_spline_multi, lower_m, upper_m, color=GREEN, alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No per-phase loss_full found in {path_loss_dir_multispline}")

# --- Single-Stage con banda (log-space) ---
if x_spline_one is not None and y_spline_one is not None:
    ymean_o, lower_o, upper_o = rolling_stats_log(y_spline_one, win=11)
    ax.plot(x_spline_one, ymean_o, label=r'S-S $\lambda_{res}=1$', color=BLUE, linewidth=2.0, zorder=3)
    ax.fill_between(x_spline_one, lower_o, upper_o, color=BLUE, alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No single phase  loss_full found in {path_loss_dir_onespline}")

# --- Single-Stage con banda (log-space) ---
if x_spline_sup is not None and y_spline_sup is not None:
    ymean_o, lower_o, upper_o = rolling_stats_log(y_spline_sup, win=11)
    ax.plot(x_spline_sup, ymean_o, label='Supervised', color='black', linewidth=2.0, zorder=3)
    ax.fill_between(x_spline_sup, lower_o, upper_o, color='black', alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No single phase  loss_full found in {path_loss_dir_splinesup}")

if plotted_any:
    ax.set_xlabel('Epoch')
    ax.set_ylabel(r'$L_2$')
    ax.set_title('PhIS-FNO')
    ax.grid(True, which='both', ls=':', alpha=0.5)
    ax.legend(loc='best')
    ax.axvline(PHASE_SPLIT_X, color='k', linewidth=0.9, dashes=(4, 3), alpha=0.9)

    # annotazioni finali (come negli altri plot: linea grigia + etichetta colorata)
    xmin, xmax = ax.get_xlim()
    margin = 0.06*(xmax - xmin)
    ax.set_xlim(xmin, xmax + margin)

    if y_spline_multi is not None:
        m, s, i0, i1 = final_plateau_stats(y_spline_multi)
        print(f"✔️  PhIS-FNO (Multi-stage) plateau: {m:.3e} ± {s:.3e} "
              f"[window: {i1-i0} iters]")

    if y_spline_one is not None:
        m, s, i0, i1 = final_plateau_stats(y_spline_one)
        print(f"✔️  PhIS-FNO (Multi-stage no reset) plateau: {m:.3e} ± {s:.3e} "
              f"[window: {i1-i0} iters]")
        
    if y_spline_sup is not None:
        m, s, i0, i1 = final_plateau_stats(y_spline_sup)
        print(f"✔️  PhIS-FNO supervised plateau: {m:.3e} ± {s:.3e} "
              f"[window: {i1-i0} iters]")

    # salvataggio
    cmp_png = os.path.join(path_loss_dir_multispline, 'multi_vs_single_errbar_log.png')
    plt.tight_layout()
    plt.savefig(cmp_png, dpi=300)
    plt.close()
    print(f"✔️  Saved comparison plot with log-space error bands in: {cmp_png}")
else:
    plt.close()

✔️  PhIS-FNO (Multi-stage) plateau: 5.124e-02 ± 3.312e-05 [window: 49 iters]
✔️  PhIS-FNO (Multi-stage no reset) plateau: 1.560e-01 ± 2.028e-04 [window: 100 iters]
✔️  PhIS-FNO supervised plateau: 9.282e-03 ± 2.088e-04 [window: 100 iters]
✔️  Saved comparison plot with log-space error bands in: loss/Spline_Multi_Stage_Adam_ep100_m24_w64_T10/multi_vs_single_errbar_log.png


In [4]:
#######################################################################
############################  MultiStage vs Single stage FNO #######
#######################################################################
import numpy as np


phases = ['P1_border_only', 'P2_res_warm']
fields = ['loss_boundary', 'loss_full', 'loss_res']

# carica le curve
x_spline_multi, y_spline_multi, last_spline_multi = load_test_full_series(path_loss_dir_multifno, phases)
x_spline_one,   y_spline_one,   last_spline_one   = load_test_full_series(path_loss_dir_onefno, phases)
x_spline_sup,   y_spline_sup,   last_spline_sup   = load_test_full_series(path_loss_dir_supervised,   phases)

BLUE  =  "#7C4DFF"
GREEN = "#00E676"

# plot
plt.figure(figsize=(8,5))
plt.yscale('log')
ax = plt.gca()
plotted_any = False

# --- Multi-Stage con banda (log-space) ---
if x_spline_multi is not None and y_spline_multi is not None:
    ymean_m, lower_m, upper_m = rolling_stats_log(y_spline_multi, win=11)
    ax.plot(x_spline_multi, ymean_m, label='M-S', color=GREEN, linewidth=2.0, zorder=3)
    ax.fill_between(x_spline_multi, lower_m, upper_m, color=GREEN, alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No per-phase loss_full found in {path_loss_dir_multispline}")

# --- Single-Stage con banda (log-space) ---
if x_spline_one is not None and y_spline_one is not None:
    ymean_o, lower_o, upper_o = rolling_stats_log(y_spline_one, win=11)
    ax.plot(x_spline_one, ymean_o, label=r'S-S $\lambda_{res}=1$', color=BLUE, linewidth=2.0, zorder=3)
    ax.fill_between(x_spline_one, lower_o, upper_o, color=BLUE, alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No single phase  loss_full found in {path_loss_dir_onespline}")

# --- Single-Stage con banda (log-space) ---
if x_spline_sup is not None and y_spline_sup is not None:
    ymean_o, lower_o, upper_o = rolling_stats_log(y_spline_sup, win=11)
    ax.plot(x_spline_sup, ymean_o, label='Supervised', color='black', linewidth=2.0, zorder=3)
    ax.fill_between(x_spline_sup, lower_o, upper_o, color='black', alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No single phase  loss_full found in {path_loss_dir_splinesup}")

if plotted_any:
    ax.set_xlabel('Epoch')
    ax.set_ylabel(r'$L_2$')
    ax.set_title('PINO')
    ax.grid(True, which='both', ls=':', alpha=0.5)
    ax.legend(loc='best')
    ax.axvline(PHASE_SPLIT_X, color='k', linewidth=0.9, dashes=(4, 3), alpha=0.9)

    # annotazioni finali (come negli altri plot: linea grigia + etichetta colorata)
    xmin, xmax = ax.get_xlim()
    margin = 0.06*(xmax - xmin)
    ax.set_xlim(xmin, xmax + margin)

    if y_spline_multi is not None:
        m, s, i0, i1 = final_plateau_stats(y_spline_multi)
        print(f"✔️  PINO (Multi-stage) plateau: {m:.3e} ± {s:.3e} "
              f"[window: {i1-i0} iters]")

    if y_spline_one is not None:
        m, s, i0, i1 = final_plateau_stats(y_spline_one)
        print(f"✔️  PINO (Multi-stage no reset) plateau: {m:.3e} ± {s:.3e} "
              f"[window: {i1-i0} iters]")
        
    if y_spline_sup is not None:
        m, s, i0, i1 = final_plateau_stats(y_spline_sup)
        print(f"✔️  PINO supervised plateau: {m:.3e} ± {s:.3e} "
              f"[window: {i1-i0} iters]")

    # salvataggio
    os.makedirs(path_loss_dir_multifno, exist_ok=True)
    cmp_png = os.path.join(path_loss_dir_multifno, 'multi_vs_single_errbar_log.png')
    plt.tight_layout()
    plt.savefig(cmp_png, dpi=300)
    plt.close()
    print(f"✔️  Saved comparison plot with log-space error bands in: {cmp_png}")
else:
    plt.close()


✔️  PINO (Multi-stage) plateau: 1.222e-01 ± 1.469e-02 [window: 20 iters]
✔️  PINO (Multi-stage no reset) plateau: 3.597e-01 ± 2.169e-02 [window: 20 iters]
✔️  PINO supervised plateau: 6.821e-03 ± 9.672e-05 [window: 90 iters]
✔️  Saved comparison plot with log-space error bands in: loss/FNO_Multi_Stage_ep100_m24_w64_T10/multi_vs_single_errbar_log.png


In [5]:
#######################################################################
############################ Boundary #################################
#######################################################################
# carica le curve
x_spline_multi, y_spline_multi, last_spline_multi = load_boundary(path_loss_dir_multispline, phases)
x_fno_multi,   y_fno_multi,   last_fno_multi     = load_boundary(path_loss_dir_multifno,   phases)

BLUE  =  "#FF2D95" 
GREEN ="#FF6D00" 

print(len(x_fno_multi))

# plot
plt.figure(figsize=(8,5))
plt.yscale('log')
ax = plt.gca()
plotted_any = False



# --- PINO (boundary) con banda log ---
if x_fno_multi is not None and y_fno_multi is not None:
    ymean_fno, lower_fno, upper_fno = rolling_stats_log(y_fno_multi, win=11)
    ax.plot(x_fno_multi, ymean_fno, label='PINO', color=GREEN, linewidth=2.0, zorder=3)
    ax.fill_between(x_fno_multi, lower_fno, upper_fno, color=GREEN, alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No per-phase boundary found in {path_loss_dir_multifno}")

# --- PhIS-FNO (boundary) con banda log ---
if x_spline_multi is not None and y_spline_multi is not None:
    ymean_spl, lower_spl, upper_spl = rolling_stats_log(y_spline_multi, win=11)
    ax.plot(x_spline_multi, ymean_spl, label='PhIS-FNO', color=BLUE, linewidth=2.0, zorder=3)
    ax.fill_between(x_spline_multi, lower_spl, upper_spl, color=BLUE, alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No per-phase boundary found in {path_loss_dir_multispline}")

if plotted_any:
    ax.set_xlabel('Epoch')
    ax.set_ylabel(r'$L_2$')
    ax.set_title('Boundary PhIS-FNO vs PINO')
    ax.grid(True, which='both', ls=':', alpha=0.5)
    ax.legend(loc='best')

    # barra verticale nera a x=200
    ax.axvline(PHASE_SPLIT_X, color='k', linewidth=0.9, dashes=(4, 3), alpha=0.9)

    # annotazioni finali (linea grigia + etichetta colorata)
    xmin, xmax = ax.get_xlim()
    margin = 0.06*(xmax - xmin)
    ax.set_xlim(xmin, xmax + margin)

    if last_fno_multi is not None:
        ax.hlines(last_fno_multi, xmin, xmax, colors=GREEN, linestyles='--', linewidth=1)
        ax.text(xmax + 0.01*(xmax-xmin), last_fno_multi,
                f"{last_fno_multi:.2e}", color=GREEN,
                va='center', ha='left', fontsize='large')
        print(f"✔️  Final boundary PINO = {last_fno_multi:.3e}")

    if last_spline_multi is not None:
        ax.hlines(last_spline_multi, xmin, xmax, colors=BLUE, linestyles='--', linewidth=1)
        ax.text(xmax + 0.01*(xmax-xmin), last_spline_multi,
                f"{last_spline_multi:.2e}", color=BLUE,
                va='center', ha='left', fontsize='large')
        print(f"✔️  Final boundary PhIS-FNO = {last_spline_multi:.3e}")

    if y_spline_multi is not None:
        m, s, i0, i1 = final_plateau_stats(y_spline_multi)
        print(f"✔️  PhIS-FNO boundary: {m:.3e} ± {s:.3e} "
              f"[window: {i1-i0} iters]")
    if y_fno_multi is not None:
        m, s, i0, i1 = final_plateau_stats(y_fno_multi)
        print(f"✔️  PINO boundary: {m:.3e} ± {s:.3e} "
              f"[window: {i1-i0} iters]")

    os.makedirs(path_loss_dir_multispline, exist_ok=True)
    cmp_png = os.path.join(path_loss_dir_multispline, 'boundary_errbar_log.png')
    plt.tight_layout()
    plt.savefig(cmp_png, dpi=300)
    plt.close()
    print(f"✔️  Saved comparison plot with log-space error bands in: {cmp_png}")
else:
    plt.close()

400
✔️  Final boundary PINO = 1.351e-02
✔️  Final boundary PhIS-FNO = 1.404e-03
✔️  PhIS-FNO boundary: 1.456e-03 ± 4.162e-05 [window: 20 iters]
✔️  PINO boundary: 1.655e-02 ± 4.790e-03 [window: 20 iters]
✔️  Saved comparison plot with log-space error bands in: loss/Spline_Multi_Stage_Adam_ep100_m24_w64_T10/boundary_errbar_log.png


In [6]:
#######################################################################
############################  MultiStage TRAINING  ####################
#######################################################################
import numpy as np


phases = ['P1_border_only', 'P2_res_warm']
fields = ['loss_boundary', 'loss_full', 'loss_res']

# carica le curve
x_fno_multi, y_fno_multi, last_fno_multi = load_test_full_series(path_loss_dir_multifno, phases)
x_spline_multi,   y_spline_multi,   last_spline_multi   = load_test_full_series(path_loss_dir_multispline, phases)

ORANGE  = "#FF6D00"  # neon orange
MAGENTA = "#FF2D95"  # neon magenta / hot pink

print(len(x_fno_multi))

# plot
plt.figure(figsize=(8,5))
plt.yscale('log')
ax = plt.gca()
plotted_any = False

# --- Multi-Stage (log-space band) ---
if x_fno_multi is not None and y_fno_multi is not None:
    ymean_m, lower_m, upper_m = rolling_stats_log(y_fno_multi, win=11)
    ax.plot(x_fno_multi, ymean_m, label='PINO', color=ORANGE, linewidth=2.0, zorder=3)
    ax.fill_between(x_fno_multi, lower_m, upper_m, color=ORANGE, alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No per-phase loss_full found in {path_loss_dir_multifno}")

# --- Single-Stage (log-space band) ---
if x_spline_multi is not None and y_spline_multi is not None:
    ymean_o, lower_o, upper_o = rolling_stats_log(y_spline_multi, win=11)
    ax.plot(x_spline_multi, ymean_o, label='PhIS-FNO', color=MAGENTA, linewidth=2.0, zorder=3)
    ax.fill_between(x_spline_multi, lower_o, upper_o, color=MAGENTA, alpha=0.18, linewidth=0, zorder=2)
    plotted_any = True
else:
    print(f"⚠️  No single phase loss_full found in {path_loss_dir_multispline}")

if plotted_any:
    ax.set_xlabel('Epoch')
    ax.set_ylabel(r'$L_2$')
    ax.set_title('Multi-Stage training')
    ax.grid(True, which='both', ls=':', alpha=0.5)
    ax.legend(loc='best')
    ax.axvline(PHASE_SPLIT_X, color='k', linewidth=0.9, dashes=(4, 3), alpha=0.9)
    # annotazioni finali (linea grigia + etichetta colorata)
    xmin, xmax = ax.get_xlim()
    margin = 0.06*(xmax - xmin)
    ax.set_xlim(xmin, xmax + margin)

    if last_fno_multi is not None:
        ax.hlines(last_fno_multi, xmin, xmax, colors=ORANGE, linestyles='--', linewidth=1)
        ax.text(xmax + 0.01*(xmax-xmin), last_fno_multi,
                f"{last_fno_multi:.2e}", color=ORANGE,
                va='center', ha='left', fontsize='large')
        print(f"✔️  Final FNO (multi) test_full = {last_fno_multi:.3e}")

    if last_spline_multi is not None:
        ax.hlines(last_spline_multi, xmin, xmax, colors=MAGENTA, linestyles='--', linewidth=1)
        ax.text(xmax + 0.01*(xmax-xmin), last_spline_multi,
                f"{last_spline_multi:.2e}", color=MAGENTA,
                va='center', ha='left', fontsize='large')
        print(f"✔️  Final FNO (single) test_full = {last_spline_multi:.3e}")

    # salvataggio
    os.makedirs(path_loss_dir_multifno, exist_ok=True)
    cmp_png = os.path.join(path_loss_dir_multispline, 'multistage_training.png')
    plt.tight_layout()
    plt.savefig(cmp_png, dpi=300)
    plt.close()
    print(f"✔️  Saved comparison plot with log-space error bands in: {cmp_png}")
else:
    plt.close()

400
✔️  Final FNO (multi) test_full = 1.143e-01
✔️  Final FNO (single) test_full = 5.121e-02
✔️  Saved comparison plot with log-space error bands in: loss/Spline_Multi_Stage_Adam_ep100_m24_w64_T10/multistage_training.png
